In [1]:
import os
import kagglehub
import shutil

path = kagglehub.competition_download('house-prices-advanced-regression-techniques')
print("Fichiers téléchargés dans :", path)

# Dossier de destination
dest = "data"
#os.makedirs(dest, exist_ok=True)

# Copier tous les fichiers du dossier téléchargé vers /data
#for fichier in os.listdir(path):
#    src_file = os.path.join(path, fichier)
#    dst_file = os.path.join(dest, fichier)
#    shutil.copy2(src_file, dst_file)

print("Fichiers copiés dans :", dest)

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fichiers téléchargés dans : /home/onyxia/.cache/kagglehub/competitions/house-prices-advanced-regression-techniques
Fichiers copiés dans : data


In [1]:
import pandas as pd
import matplotlib as plt
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import importlib
import preprocessing as p
importlib.reload(p)

df_train=pd.read_csv('data/train.csv')
df_test=pd.read_csv('data/test.csv')

In [2]:
X=df_train.drop(columns=['SalePrice'])
y=df_train['SalePrice']
X_train,X_valid,y_train,y_valid=train_test_split(X,y,test_size=0.2,random_state=42)

In [3]:
test_ids = df_test['Id'].copy()
X_train,power_transform,imputer=p.preprocessing(X_train)
X_valid,_,_=p.preprocessing(X_valid,power_transform,imputer)
df_test,_,_=p.preprocessing(df_test,power_transform,imputer)

In [4]:
model = LinearRegression()
model.fit(X_train, np.log1p(y_train))

y_pred = model.predict(X_valid)
rmse = np.sqrt(mean_squared_error(np.log1p(y_valid), y_pred))
print(rmse)

0.43156499920542035


In [5]:
erreurs = np.abs(y_pred - np.log1p(y_valid).values)
idx = np.argsort(erreurs)[-5:]   # les 5 pires
print("pred :", y_pred[idx])
print("vrai :", np.log1p(y_valid).values[idx])
print("écart:", erreurs[idx])

print(X_valid.describe().loc[['min','max']].T.sort_values('max').tail())
print(X_valid.describe().loc[['min','max']].T.sort_values('min').head())

print(X_train['Utilities'].max(), X_train['Functional'].max())

pred : [12.04634757 12.04634757 12.04634757 12.04634757 12.04634757]
vrai : [13.22956979 13.32392858 10.59665973 13.53447435 10.47197813]
écart: [1.18322222 1.27758102 1.44968783 1.48812679 1.57436944]
                     min           max
YrSold      2.006000e+03  2.010000e+03
BsmtUnfSF   0.000000e+00  2.042000e+03
LotArea     1.491000e+03  7.076100e+04
PavedDrive  5.660015e+04  6.445289e+10
Functional  4.972361e+25  4.984177e+41
                   min        max
MasVnrArea   -31.02874   3.821645
BsmtQual       0.00000  17.212464
BsmtFinType1   0.00000   6.000000
BsmtExposure   0.00000   1.042523
FullBath       0.00000   3.000000
4.0 4.984176749671765e+41


In [6]:
import inspect
print(inspect.getsource(p.preprocessing))

def preprocessing(df,power_transformer=None,imputer=None):
    df = df.drop(columns=['Id'])
    
    #On fill les NaN
    cat_cols = df.select_dtypes(exclude='number').columns
    df[cat_cols] = df[cat_cols].fillna('None')

    #On crée une catégories Pool (apporte +d'infos)
    df['Pool']=(df['PoolArea']>0).astype(float)
    df = df.drop(columns=['PoolArea'])

    #Modification des variables catégorielles qui sont en fait ordinales

    # Échelle de qualité générique (revient sur 6 colonnes)
    qual_scale = ['Po', 'Fa', 'TA', 'Gd', 'Ex']

    # Variantes avec "pas d'objet" (NA) en plus, à placer selon ton choix
    # (toi tu as déjà remplacé les NA par 'None' plus tôt dans la conversation)
    qual_scale_with_none = ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']

    # PoolQC n'a pas de modalité 'Po' dans le fichier
    poolqc_scale = ['None', 'Fa', 'TA', 'Gd', 'Ex']

    ordinal_scales = {
        'ExterQual':    qual_scale_with_none,   # pas de NA en réalité, mais safe
        'ExterCond':

In [7]:
print("SimpleImputer" in inspect.getsource(p.preprocessing))

False
